# Chapter 5 — Self-Sufficient Notebook: Autonomous Customer Service Agent

This notebook converts the chapter's most important architecture snippet into a runnable, self-contained Jupyter notebook.

## Why this snippet
I selected the **Autonomous Customer Service Agent cognitive loop** because it best captures the chapter's core idea:
- **perception**
- **reasoning**
- **action**
- **learning**

It is the clearest end-to-end example of a foundational cognitive architecture in motion.

The notebook is fully self-sufficient:
- no API key required
- all helper functions are mocked locally
- you can run the full loop from a sample support request

## What this notebook demonstrates

- Enhanced perception with situational context
- Autonomous reasoning and strategy selection
- Action planning with simple workflow selection
- Safety and escalation checks
- Learning from interaction outcomes
- Session memory updates

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List

## Step 1 — Mock helper functions

The book chapter uses illustrative pseudocode with external services like monitoring systems, CRM tools, and human agent availability checks.

Here we replace those with lightweight local functions so the architecture is runnable.

In [ ]:
def classify_text_sentiment(text: str) -> str:
    text_l = text.lower()
    negative_cues = ["down", "can't", "cannot", "intermittent", "urgent", "critical", "problem"]
    if any(word in text_l for word in negative_cues):
        return "negative"
    return "neutral"

def monitoring_client_get_active_alerts() -> List[str]:
    return ["regional_outage_east", "dns_instability_notice"]

def crm_client_get_account_tier(user_id: str) -> str:
    if "premium" in user_id or "business" in user_id:
        return "premium"
    return "standard"

def check_human_agent_status() -> bool:
    return True

def classify_intent(message: str) -> str:
    text = message.lower()
    if "billing" in text or "charge" in text:
        return "billing_issue"
    if "internet" in text or "service" in text or "outage" in text or "connect" in text:
        return "service_outage"
    return "general_support"

def get_user_history(user_id: str) -> Dict[str, Any]:
    return {"past_tickets": 3, "recent_satisfaction": 0.82}

def determine_priority(intent: str, sentiment: str, user_history: Dict[str, Any]) -> str:
    if intent == "service_outage" and sentiment == "negative":
        return "high"
    if intent == "billing_issue":
        return "medium"
    return "low"

def determine_autonomy_level(user_tier: str, current_load: int, agent_availability: bool) -> float:
    score = 0.6
    if user_tier == "premium":
        score += 0.15
    if current_load > 50:
        score += 0.1
    if not agent_availability:
        score += 0.1
    return min(score, 1.0)

def calculate_escalation_threshold(time_of_day: int, intent: str, priority: str) -> float:
    base = 0.6
    if priority == "high":
        base -= 0.15
    if intent == "service_outage":
        base -= 0.1
    if time_of_day >= 20 or time_of_day <= 6:
        base += 0.05
    return max(0.2, min(base, 0.95))

def calculate_success(results: List[str], decision_context: Dict[str, Any]) -> float:
    strategy = decision_context.get("strategy", "")
    if strategy == "full_autonomous_resolution" and len(results) >= 4:
        return 0.9
    if strategy == "guided_autonomous_resolution":
        return 0.8
    return 0.7

def update_user_preferences(user_id: str, success_score: float) -> None:
    print(f"[Learning] Updated user preference profile for {user_id} with success={success_score:.2f}")

def flag_for_model_improvement(perception_data: Dict[str, Any]) -> None:
    print(f"[Learning] Flagged interaction for review: {perception_data['message'][:60]}")

def check_escalation_preference(user_id: str) -> bool:
    return False

## Step 2 — Perception layer

In [ ]:
def analyze_sentiment(text: str) -> str:
    return classify_text_sentiment(text)

def check_system_alerts() -> List[str]:
    return monitoring_client_get_active_alerts()

def get_user_tier(user_id: str) -> str:
    return crm_client_get_account_tier(user_id)

def perceive_input(user_message: str, context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "message": user_message,
        "timestamp": datetime.now(),
        "user_id": context.get("user_id"),
        "session_state": context.get("session"),
        "sentiment": analyze_sentiment(user_message),
    }

def enhanced_perception(user_message: str, context: Dict[str, Any], system_state: Dict[str, Any]) -> Dict[str, Any]:
    base_perception = perceive_input(user_message, context)
    enhanced_data = {
        **base_perception,
        "current_load": system_state.get("active_sessions", 0),
        "time_of_day": datetime.now().hour,
        "user_tier": get_user_tier(context.get("user_id", "")),
        "recent_issues": check_system_alerts(),
        "agent_availability": check_human_agent_status(),
    }
    return enhanced_data

## Step 3 — Reasoning layer

In [ ]:
def autonomous_reasoning(perception_data: Dict[str, Any]) -> Dict[str, Any]:
    intent = classify_intent(perception_data["message"])
    priority = determine_priority(
        intent,
        perception_data["sentiment"],
        user_history=get_user_history(perception_data["user_id"]),
    )

    autonomy_level_score = determine_autonomy_level(
        perception_data["user_tier"],
        perception_data["current_load"],
        perception_data["agent_availability"],
    )
    urgency_score = 0.9 if priority == "high" else 0.5 if priority == "medium" else 0.2
    complexity_score = 0.7 if intent == "billing_issue" else 0.4 if intent == "service_outage" else 0.3
    escalation_threshold = calculate_escalation_threshold(
        perception_data["time_of_day"],
        intent,
        priority,
    )

    decision_context = {
        "intent": intent,
        "priority": priority,
        "context": perception_data,
        "autonomy_level_score": autonomy_level_score,
        "urgency_score": urgency_score,
        "complexity_score": complexity_score,
        "agent_availability": perception_data["agent_availability"],
        "escalation_threshold": escalation_threshold,
    }

    strategy_scores = {
        "full_autonomous_resolution":
            decision_context["autonomy_level_score"] * 0.5
            + (1 - decision_context["escalation_threshold"]) * 0.3
            + (1 - decision_context["complexity_score"]) * 0.2,
        "immediate_escalation":
            decision_context["urgency_score"] * 0.4
            + (0 if decision_context["agent_availability"] else 1) * 0.4
            + decision_context["escalation_threshold"] * 0.2,
        "guided_autonomous_resolution": 0.5,
    }

    decision_context["strategy"] = max(strategy_scores, key=strategy_scores.get)
    decision_context["strategy_scores"] = strategy_scores
    return decision_context

## Step 4 — Action planning and execution

In [ ]:
def create_autonomous_plan(decision_context: Dict[str, Any]) -> List[Dict[str, Any]]:
    intent = decision_context["intent"]

    if intent == "billing_issue":
        return [
            {"id": "T1", "action": "fetch_account_details", "depends_on": []},
            {"id": "T2", "action": "analyze_billing_history", "depends_on": ["T1"]},
            {"id": "T3", "action": "identify_discrepancies", "depends_on": ["T2"]},
            {"id": "T4", "action": "generate_explanation", "depends_on": ["T3"]},
        ]
    elif intent == "service_outage":
        return [
            {"id": "T1", "action": "check_service_status", "depends_on": []},
            {"id": "T2", "action": "identify_affected_areas", "depends_on": ["T1"]},
            {"id": "T3", "action": "estimate_resolution_time", "depends_on": ["T1"]},
            {"id": "T4", "action": "send_proactive_notifications", "depends_on": ["T2", "T3"]},
            {"id": "T5", "action": "schedule_follow_up", "depends_on": ["T4"]},
        ]

    return [
        {"id": "T1", "action": "analyze_issue", "depends_on": []},
        {"id": "T2", "action": "research_solutions", "depends_on": ["T1"]},
        {"id": "T3", "action": "implement_resolution", "depends_on": ["T2"]},
    ]

def execute_autonomous_workflow(action_plan: List[Dict[str, Any]], decision_context: Dict[str, Any]) -> List[str]:
    results = []
    for step in action_plan:
        results.append(f"Executed {step['action']}")
    return results

def prepare_escalation_package(decision_context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "summary": f"Escalate {decision_context['intent']} with priority {decision_context['priority']}",
        "context": decision_context["context"],
    }

def initiate_human_handoff(escalation_data: Dict[str, Any]) -> List[str]:
    return [f"Escalated to human agent: {escalation_data['summary']}"]

def execute_with_checkpoints(action_plan: List[Dict[str, Any]], decision_context: Dict[str, Any]) -> List[str]:
    results = []
    for step in action_plan:
        results.append(f"Checkpoint complete for {step['action']}")
    return results

def analyze_execution_results(results: List[str], decision_context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "success_ratio": min(1.0, len(results) / 5),
        "strategy_used": decision_context["strategy"],
    }

def update_decision_models(execution_feedback: Dict[str, Any]) -> None:
    print(f"[Adaptation] Updated decision model from strategy={execution_feedback['strategy_used']} "
          f"success_ratio={execution_feedback['success_ratio']:.2f}")

def autonomous_action_execution(decision_context: Dict[str, Any]) -> List[str]:
    strategy = decision_context["strategy"]

    if strategy == "full_autonomous_resolution":
        action_plan = create_autonomous_plan(decision_context)
        results = execute_autonomous_workflow(action_plan, decision_context)
    elif strategy == "immediate_escalation":
        escalation_data = prepare_escalation_package(decision_context)
        results = initiate_human_handoff(escalation_data)
    else:
        action_plan = create_autonomous_plan(decision_context)
        results = execute_with_checkpoints(action_plan, decision_context)

    execution_feedback = analyze_execution_results(results, decision_context)
    update_decision_models(execution_feedback)
    return results

## Step 5 — Safety and escalation

In [ ]:
def calculate_credit_value(decision_context: Dict[str, Any]) -> float:
    return 150.0

def get_autonomous_credit_limit(user_tier: str) -> float:
    return 250.0 if user_tier == "premium" else 50.0

def requires_sensitive_data_access(planned_actions: List[str]) -> bool:
    sensitive_actions = {"fetch_account_details", "analyze_billing_history"}
    return any(action in sensitive_actions for action in planned_actions)

def verify_autonomous_data_permissions(decision_context: Dict[str, Any]) -> bool:
    return decision_context["context"]["user_tier"] == "premium"

def assess_customer_impact_risk(planned_actions: List[str]) -> float:
    return 0.5 if "schedule_follow_up" in planned_actions else 0.3

def autonomous_safety_check(decision_context: Dict[str, Any], planned_actions: List[str]):
    safety_violations = []

    if "apply_account_credits" in planned_actions:
        credit_amount = calculate_credit_value(decision_context)
        if credit_amount > get_autonomous_credit_limit(decision_context["context"]["user_tier"]):
            safety_violations.append("credit_limit_exceeded")

    if requires_sensitive_data_access(planned_actions):
        if not verify_autonomous_data_permissions(decision_context):
            safety_violations.append("insufficient_data_permissions")

    if assess_customer_impact_risk(planned_actions) > decision_context["escalation_threshold"]:
        safety_violations.append("high_impact_risk")

    return len(safety_violations) == 0, safety_violations

def assess_issue_complexity(decision_context: Dict[str, Any]) -> float:
    return decision_context.get("complexity_score", 0.5)

def calculate_business_impact(decision_context: Dict[str, Any]) -> float:
    return 0.8 if decision_context["priority"] == "high" else 0.4

def calculate_weighted_escalation_score(escalation_factors: Dict[str, bool]) -> float:
    weights = {
        "safety_violations": 0.30,
        "confidence_threshold": 0.20,
        "complexity_threshold": 0.15,
        "user_preference": 0.10,
        "business_impact": 0.25,
    }
    return sum(weights[k] * int(v) for k, v in escalation_factors.items())

def prepare_intelligent_escalation(decision_context: Dict[str, Any], escalation_factors: Dict[str, bool]) -> Dict[str, Any]:
    return {
        "decision_context": decision_context,
        "factors": escalation_factors,
        "recommended_action": "human_review",
    }

def intelligent_escalation_decision(decision_context: Dict[str, Any], safety_results):
    escalation_factors = {
        "safety_violations": not safety_results[0],
        "confidence_threshold": 0.9 < 0.8,  # mocked confidence check
        "complexity_threshold": assess_issue_complexity(decision_context) > 0.7,
        "user_preference": check_escalation_preference(decision_context["context"]["user_id"]),
        "business_impact": calculate_business_impact(decision_context) > 0.6,
    }

    escalation_score = calculate_weighted_escalation_score(escalation_factors)

    if escalation_score > decision_context["escalation_threshold"]:
        return prepare_intelligent_escalation(decision_context, escalation_factors)
    return None

## Step 6 — Autonomous agent class

In [ ]:
class AutonomousCustomerServiceAgent:
    def __init__(self):
        self.session_memory: Dict[str, Any] = {}
        self.decision_history: List[Dict[str, Any]] = []
        self.performance_metrics: Dict[str, Any] = {}

    def get_current_system_state(self) -> Dict[str, Any]:
        return {"active_sessions": 64}

    def update_session_memory(self, user_id: str, decision_context: Dict[str, Any], results: List[str]) -> None:
        self.session_memory[user_id] = {
            "last_strategy": decision_context["strategy"],
            "last_intent": decision_context["intent"],
            "last_updated": datetime.now().isoformat(),
            "result_count": len(results),
        }

    def adjust_autonomy_thresholds(self, decision_context: Dict[str, Any], success_score: float) -> None:
        print(f"[Learning] Adjusted autonomy thresholds after success={success_score:.2f}")

    def update_strategy_effectiveness(self, strategy: str, success_score: float) -> None:
        self.performance_metrics[strategy] = success_score

    def learn_from_interaction(self, perception_data: Dict[str, Any], decision_context: Dict[str, Any], results: List[str]):
        success_score = calculate_success(results, decision_context)

        update_user_preferences(perception_data["user_id"], success_score)

        if success_score < 0.7:
            self.adjust_autonomy_thresholds(decision_context, success_score)
            flag_for_model_improvement(perception_data)

        self.update_strategy_effectiveness(decision_context["strategy"], success_score)

        self.decision_history.append({
            "perception": perception_data,
            "decision": decision_context,
            "outcome": success_score,
            "timestamp": datetime.now(),
        })

    def cognitive_loop(self, user_message: str, context: Dict[str, Any]) -> Dict[str, Any]:
        system_state = self.get_current_system_state()

        perception_data = enhanced_perception(user_message, context, system_state)
        decision_context = autonomous_reasoning(perception_data)

        planned_steps = create_autonomous_plan(decision_context)
        planned_actions = [step["action"] for step in planned_steps]
        safety_check, violations = autonomous_safety_check(decision_context, planned_actions)
        escalation = intelligent_escalation_decision(decision_context, (safety_check, violations))

        if escalation:
            results = [f"Escalated after intelligent review with factors: {escalation['factors']}"]
        else:
            results = autonomous_action_execution(decision_context)

        self.learn_from_interaction(perception_data, decision_context, results)
        self.update_session_memory(context.get("user_id"), decision_context, results)

        return {
            "perception": perception_data,
            "decision": decision_context,
            "safety": {"passed": safety_check, "violations": violations},
            "results": results,
            "session_memory": self.session_memory.get(context.get("user_id")),
        }

## Step 7 — Run the example from the chapter

In [ ]:
agent = AutonomousCustomerServiceAgent()

user_message = "My business internet has been intermittent for two days, and I have a critical presentation tomorrow."
context = {"user_id": "premium_business_123", "session": "new"}

run_output = agent.cognitive_loop(user_message, context)
run_output

## Step 8 — Read the result more clearly

In [ ]:
print("PERCEPTION")
print("-" * 60)
for k, v in run_output["perception"].items():
    print(f"{k}: {v}")

print("\nDECISION")
print("-" * 60)
for k, v in run_output["decision"].items():
    if k != "context":
        print(f"{k}: {v}")

print("\nSAFETY")
print("-" * 60)
print(run_output["safety"])

print("\nRESULTS")
print("-" * 60)
for item in run_output["results"]:
    print("-", item)

print("\nSESSION MEMORY")
print("-" * 60)
print(run_output["session_memory"])

## Why this is the chapter's key example

This notebook captures the chapter's central engineering message:

A foundational agent is not just a single model call.  
It is a **loop** with:
1. structured perception,
2. explicit reasoning,
3. controlled execution,
4. feedback-driven learning.

That is the backbone the later architectures build on.

## Suggested extensions

- replace the mocked classifiers with a real LLM
- add persistent storage for session memory
- add a planning module for multi-step dependency execution
- add a vector store to evolve this into a memory-augmented agent